In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Extreme Model Triage Benchmark Cascade (`models/multi_extreme.ipynb`)

This notebook benchmarks the **Extreme Model Triage Cascade Architecture** based on `TODO.md` structure using pre-trained model artifacts from `models/lr_extreme.ipynb` and `models/rf_extreme.ipynb`:

### Decision Logic
1. **Extreme Model (Layer 1)**: Loaded from `deploy/lr_extreme_model.rds` or `deploy/rf_extreme_model.rds`.
   - Evaluates sample and predicts `"1"` (ESI 1), `"5"` (ESI 5), or `"neither"` (ESI 2, 3, 4).
   - `if (Layer 1 output == "1")` -> Classify **ESI 1**.
   - `else if (Layer 1 output == "5")` -> Classify **ESI 5**.
2. **Intermediate Acuity Model (Layer 2)**: Loaded from `deploy/rf_feng_esi234_extreme_model.rds` (or `deploy/lr_feng_esi234_extreme_model.rds`).
   - `else` (`"neither"`) -> Classify intermediate acuity (**ESI 2**, **ESI 3**, or **ESI 4**).

### Evaluation Protocol
- Evaluated on the **Complete Test Distribution Set** from `.RData`.
- Reports **5x5 Confusion Matrix**, **Accuracy**, **Precision**, **Recall (Sensitivity)**, **PR-AUC**, **Multi-Class ROC-AUC**, and target class comparison tables.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(randomForest)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Complete .RData Dataset & Construct Complete Case Features
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading complete dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

base_features <- config$features$base_features
if (is.null(base_features)) base_features <- config$features$data_name

df <- raw_df[, c(intersect(base_features, names(raw_df)), target_col)]
if ("gender" %in% names(df)) df$gender <- ifelse(as.character(df$gender) == "Male", 1, 0)

df <- na.omit(df)
raw_esi <- as.character(df[[target_col]])
df$raw_esi <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))

cat(sprintf("Complete Case Test Population Ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("5-Class ESI Distribution:\n")
print(table(df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Extract Complete Case Test Partition (15% Test Set)
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
in_train_val <- createDataPartition(df$raw_esi, p = 1 - test_size, list = FALSE)
test_df      <- df[-in_train_val, ]

cat(sprintf("Complete Case Test Partition: %d rows\n", nrow(test_df)))
cat("Test Set Class Distribution:\n")
print(table(test_df$raw_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Read Pre-Trained Model Artifacts (LR Extreme, RF Extreme & Intermediate RF ESI 234)
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"

path_lr_ext <- file.path(deploy_dir, "lr_extreme_model.rds")
path_rf_ext <- file.path(deploy_dir, "rf_extreme_model.rds")
path_rf_234 <- file.path(deploy_dir, "rf_feng_esi234_extreme_model.rds")

cat("Reading pre-trained model artifacts from:", deploy_dir, "...\n")
mod_lr_ext <- if (file.exists(path_lr_ext)) readRDS(path_lr_ext) else NULL
mod_rf_ext <- if (file.exists(path_rf_ext)) readRDS(path_rf_ext) else NULL
mod_rf_234 <- if (file.exists(path_rf_234)) readRDS(path_rf_234) else NULL

cat(sprintf("Loaded Artifact Status:\n  - LR Extreme Model (lr_extreme_model.rds)         : %s\n  - RF Extreme Model (rf_extreme_model.rds)         : %s\n  - RF ESI 2,3,4 Model (rf_feng_esi234_extreme_model.rds) : %s\n",
            !is.null(mod_lr_ext), !is.null(mod_rf_ext), !is.null(mod_rf_234)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Extreme Cascade Decision Engine & Prediction Logic
# ---------------------------------------------------------
predict_extreme_cascade <- function(data, extreme_mod, intermed_mod) {
  N <- nrow(data)
  
  # 1. Preprocess for Extreme Model
  d_ext <- data
  if (!is.null(extreme_mod$preproc)) {
    num_cols <- names(d_ext)[sapply(d_ext, is.numeric)]
    d_ext[, num_cols] <- predict(extreme_mod$preproc, d_ext[, num_cols])
  }
  
  # 2. Preprocess for Intermediate Model
  d_int <- data
  if (!is.null(intermed_mod$preproc)) {
    num_cols_int <- intersect(names(intermed_mod$preproc$mean), names(d_int))
    if (length(num_cols_int) > 0) {
      d_int[, num_cols_int] <- predict(intermed_mod$preproc, d_int[, num_cols_int, drop = FALSE])
    }
  }
  
  # 3. Direct Extreme Model Prediction ('1', '5', 'neither')
  pred_ext <- as.character(predict(extreme_mod$model, newdata = d_ext))
  prob_ext <- predict(extreme_mod$model, newdata = d_ext, type = "prob")
  if (is.null(prob_ext)) prob_ext <- predict(extreme_mod$model, newdata = d_ext, type = "probs")
  
  # 4. Direct Intermediate Model Prediction ('2', '3', '4')
  d_int_rf <- d_int[, setdiff(names(d_int), c("raw_esi", "target_esi234", "target_layer1"))]
  d_int_rf <- na.roughfix(d_int_rf)
  
  pred_int <- as.character(predict(intermed_mod$model, newdata = d_int_rf, type = "response"))
  prob_int <- predict(intermed_mod$model, newdata = d_int_rf, type = "prob")
  
  # Construct 5-Class Probabilities
  p1 <- if (is.matrix(prob_ext) && "1" %in% colnames(prob_ext)) prob_ext[, "1"] else rep(0, N)
  p5 <- if (is.matrix(prob_ext) && "5" %in% colnames(prob_ext)) prob_ext[, "5"] else rep(0, N)
  
  p2 <- if (is.matrix(prob_int) && "2" %in% colnames(prob_int)) prob_int[, "2"] else rep(0, N)
  p3 <- if (is.matrix(prob_int) && "3" %in% colnames(prob_int)) prob_int[, "3"] else rep(0, N)
  p4 <- if (is.matrix(prob_int) && "4" %in% colnames(prob_int)) prob_int[, "4"] else rep(0, N)
  
  prob_matrix <- cbind("1" = p1, "2" = p2, "3" = p3, "4" = p4, "5" = p5)
  
  # Sequential Decision Rule: '1' -> ESI 1; '5' -> ESI 5; 'neither' -> Intermediate RF ('2', '3', '4')
  pred_classes <- character(N)
  for (i in 1:N) {
    if (pred_ext[i] == "1") {
      pred_classes[i] <- "1"
    } else if (pred_ext[i] == "5") {
      pred_classes[i] <- "5"
    } else {
      pred_classes[i] <- pred_int[i]  # "2", "3", or "4"
    }
  }
  
  pred_factor <- factor(pred_classes, levels = c("1", "2", "3", "4", "5"))
  return(list(pred_factor = pred_factor, prob_matrix = prob_matrix))
}

cat("Extreme Cascade Decision Engine Initialized!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Benchmark Cascade Variant A (LR Extreme + Intermediate RF ESI 234)
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_cascade_variant <- function(extreme_mod, intermed_mod, test_data, cascade_name) {
  if (is.null(extreme_mod) || is.null(intermed_mod)) {
    cat(sprintf("Skipping %s: Required model artifacts not found.\n", cascade_name))
    return(NULL)
  }
  
  res <- predict_extreme_cascade(test_data, extreme_mod, intermed_mod)
  pred_test <- res$pred_factor
  prob_test <- res$prob_matrix
  actual_test <- factor(test_data$raw_esi, levels = c("1", "2", "3", "4", "5"))
  
  cm  <- confusionMatrix(pred_test, actual_test)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- cm$byClass[, "Pos Pred Value"]
  rec_by_class  <- cm$byClass[, "Sensitivity"]
  macro_prec    <- mean(prec_by_class, na.rm = TRUE)
  macro_rec     <- mean(rec_by_class,  na.rm = TRUE)
  
  pr_auc_by_class <- numeric(5)
  names(pr_auc_by_class) <- c("1", "2", "3", "4", "5")
  for (cls in c("1", "2", "3", "4", "5")) {
    act_bin <- ifelse(actual_test == cls, 1, 0)
    pr_auc_by_class[cls] <- calc_pr_auc(act_bin, prob_test[, cls])
  }
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  roc_obj <- pROC::multiclass.roc(actual_test, prob_test)
  macro_roc_auc <- as.numeric(roc_obj$auc)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   %s - COMPLETE TEST SET BENCHMARK\n", toupper(cascade_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy        : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision         : %.4f (%.2f%%)\n", macro_prec, macro_prec * 100))
  cat(sprintf("  Macro Recall (Sens)     : %.4f (%.2f%%)\n", macro_rec, macro_rec * 100))
  cat(sprintf("  Macro PR-AUC            : %.4f\n", macro_pr_auc))
  cat(sprintf("  Multi-Class ROC-AUC     : %.4f\n", macro_roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Per-Class Performance Summary Table:\n")
  per_class_metrics <- data.frame(
    Class        = c("1", "2", "3", "4", "5"),
    Actual_Count = as.numeric(table(actual_test)),
    Pred_Count   = as.numeric(table(pred_test)),
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  print(per_class_metrics)
  
  cat("\nFull 5x5 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  return(per_class_metrics)
}
cat("--- EVALUATING CASCADE VARIANT A: LR EXTREME + INTERMEDIATE RF ---\n")
metrics_varA <- evaluate_cascade_variant(mod_lr_ext, mod_rf_234, test_df, "Cascade Variant A (LR Extreme + Intermediate RF)")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Benchmark Cascade Variant B (RF Extreme + Intermediate RF ESI 234)
# ---------------------------------------------------------
cat("--- EVALUATING CASCADE VARIANT B: RF EXTREME + INTERMEDIATE RF ---\n")
metrics_varB <- evaluate_cascade_variant(mod_rf_ext, mod_rf_234, test_df, "Cascade Variant B (RF Extreme + Intermediate RF)")